# 03 - Leitura e Sintese de Papers Cientificos
> Use o Claude para extrair informacoes-chave de artigos, metodologias e gaps na literatura

**Modulo:** `10_engfis_academico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


## Setup Inicial

Configuracao padrao do cliente Anthropic e funcao auxiliar.

In [ ]:
from anthropic import Anthropic

client = Anthropic()

HAIKU = "claude-haiku-4-20250414"
SONNET = "claude-sonnet-4-20250514"


def ask(prompt, system="", model=SONNET, max_tokens=4096):
    """Envia uma mensagem ao Claude e retorna a resposta."""
    messages = [{"role": "user", "content": prompt}]
    kwargs = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text


print("Cliente configurado com sucesso!")

---

## Resumindo Abstracts Tecnicos

Uma das tarefas mais frequentes na vida academica e ler dezenas de abstracts
para decidir quais papers merecem leitura completa. Vamos usar o Claude para
extrair informacoes estruturadas de um abstract tecnico.

In [ ]:
# Abstract ficticio mas realista sobre caracterizacao de filmes finos
abstract = """
Thin films of titanium dioxide (TiO2) were deposited on silicon (100) substrates
using radio-frequency magnetron sputtering at varying substrate temperatures
(200-500 C). The structural, morphological, and optical properties were
systematically investigated using X-ray diffraction (XRD), atomic force microscopy
(AFM), and UV-Vis spectrophotometry. XRD analysis revealed a phase transition from
amorphous to anatase structure at 300 C, with rutile phase appearing above 450 C.
AFM measurements showed that surface roughness increased from 1.2 nm to 8.7 nm
with increasing substrate temperature. The optical band gap, determined by Tauc
plot analysis, varied from 3.4 eV (amorphous) to 3.2 eV (anatase), consistent
with literature values. Films deposited at 350 C exhibited the highest
photocatalytic activity under UV irradiation, attributed to the optimal
combination of anatase crystallinity and moderate surface roughness. These results
suggest that substrate temperature is a critical parameter for tailoring TiO2 thin
film properties for photocatalytic applications. However, long-term stability
under continuous UV exposure was not assessed in this study.
"""

prompt = f"""Analise o seguinte abstract de um paper cientifico e forneca um resumo
estruturado com exatamente estes campos:

1. **Objetivo**: O que os autores buscaram investigar?
2. **Metodo**: Quais tecnicas experimentais foram usadas?
3. **Resultados-chave**: Liste os 3 principais achados numericos.
4. **Conclusao**: Qual a principal conclusao?
5. **Limitacoes**: O que NAO foi abordado no estudo?
6. **Relevancia para Engenharia Fisica**: Por que isso importa?

Abstract:
{abstract}
"""

resumo = ask(prompt)
print(resumo)

### Processamento em lote de abstracts

Na pratica, voce vai querer processar varios abstracts de uma vez para
fazer triagem rapida.

In [ ]:
# Lista de abstracts para triagem rapida
abstracts = [
    {
        "titulo": "Efeito da dopagem com nitrogenio em TiO2",
        "texto": "Nitrogen-doped TiO2 thin films were prepared by reactive sputtering. "
                 "The N-doping shifted the absorption edge to 2.8 eV, enabling visible "
                 "light photocatalysis. Degradation of methylene blue reached 95% under "
                 "solar simulator irradiation after 2 hours."
    },
    {
        "titulo": "Nanoestruturas de ZnO para sensores de gas",
        "texto": "ZnO nanorods were grown on glass substrates via hydrothermal synthesis. "
                 "The sensors showed high sensitivity to NO2 at concentrations as low as "
                 "50 ppb at room temperature, with response times below 30 seconds."
    },
    {
        "titulo": "Perovskitas hibridas para celulas solares",
        "texto": "Methylammonium lead iodide perovskite films were fabricated using a "
                 "two-step deposition method. Power conversion efficiency of 18.3% was "
                 "achieved. However, devices showed 40% degradation after 500 hours of "
                 "continuous illumination under ambient conditions."
    }
]

prompt_triagem = """Para cada abstract abaixo, forneca uma classificacao de
relevancia (Alta/Media/Baixa) para pesquisa em materiais para energia solar,
junto com uma justificativa em UMA frase.

Formato de saida para cada paper:
- **Titulo**: ...
- **Relevancia**: Alta/Media/Baixa
- **Justificativa**: ...

Abstracts:
"""

for i, paper in enumerate(abstracts, 1):
    prompt_triagem += f"\n{i}. {paper['titulo']}:\n{paper['texto']}\n"

triagem = ask(prompt_triagem)
print(triagem)

---

## Extraindo Metodologia de um Paper

Quando voce encontra um paper com uma tecnica experimental que quer replicar,
extrair um protocolo passo-a-passo e extremamente util. Vamos simular isso
com uma descricao de procedimento experimental.

In [ ]:
# Texto descrevendo um procedimento experimental de XRD
texto_metodologia = """
The thin film samples were prepared for X-ray diffraction analysis following
standard procedures. Prior to measurement, each substrate was cleaned using
an ultrasonic bath in acetone for 15 minutes, followed by isopropanol for
10 minutes, and finally dried with nitrogen gas flow. The films were deposited
using RF magnetron sputtering from a 99.99% pure TiO2 target at a base
pressure of 5e-7 Torr. Argon was used as the sputtering gas at a flow rate
of 20 sccm, maintaining a working pressure of 5 mTorr. The RF power was set
to 150 W and deposition time was 2 hours for all samples.

XRD measurements were performed on a Bruker D8 Advance diffractometer using
Cu K-alpha radiation (lambda = 1.5406 Angstroms) operated at 40 kV and 40 mA.
Theta-2theta scans were recorded in the range of 20-80 degrees with a step
size of 0.02 degrees and a counting time of 2 seconds per step. Grazing
incidence XRD (GIXRD) was also performed at an incidence angle of 0.5 degrees
to enhance surface sensitivity. Phase identification was carried out using
the ICDD PDF-4+ database. Crystallite sizes were estimated from peak broadening
using the Scherrer equation after correcting for instrumental broadening using
a LaB6 standard reference material.
"""

prompt_metodo = f"""Voce e um especialista em caracterizacao de materiais. Extraia
do texto abaixo um protocolo experimental passo-a-passo detalhado.

Para cada etapa, inclua:
- Numero do passo
- Descricao da acao
- Parametros especificos (temperatura, tempo, pressao, etc.)
- Equipamento necessario

Separe em duas secoes:
1. **Preparacao da Amostra**
2. **Medicao por XRD**

Ao final, liste os parametros criticos que NAO devem ser alterados sem
justificativa.

Texto do paper:
{texto_metodologia}
"""

protocolo = ask(prompt_metodo)
print(protocolo)

---

## Comparando Multiplos Papers

Em uma revisao bibliografica, e essencial comparar diferentes abordagens
para o mesmo problema. O Claude pode criar tabelas comparativas
que facilitam a analise.

In [ ]:
# Resumos de 3 papers sobre melhoria de eficiencia de celulas solares
papers_comparacao = """
Paper 1 - "Interface Engineering in Perovskite Solar Cells" (Zhang et al., 2023)
Abordagem: Modificacao da interface ETL/perovskita com SAMs (Self-Assembled Monolayers)
Material: MAPbI3 com camada de [2-(9H-carbazol-9-yl)ethyl]phosphonic acid
Eficiencia: 21.4% (vs 18.7% referencia)
Estabilidade: 90% da eficiencia inicial apos 1000h sob iluminacao continua
Custo estimado: Medio (SAMs sao relativamente baratos)
Escalabilidade: Alta (compativel com deposicao em larga escala)

Paper 2 - "Tandem Si/Perovskite Architecture" (Mueller et al., 2023)
Abordagem: Celula tandem monolitica silicio/perovskita
Material: Cs0.05FA0.95PbI3 sobre celula de Si heterojuncao
Eficiencia: 29.1% (vs 25.7% Si sozinho)
Estabilidade: 85% apos 500h (degradacao na camada perovskita)
Custo estimado: Alto (processo complexo de fabricacao)
Escalabilidade: Media (alinhamento de corrente e desafiador)

Paper 3 - "Quantum Dot Sensitized Absorption" (Park et al., 2023)
Abordagem: Incorporacao de quantum dots de PbS na camada ativa
Material: PbS QDs (3.2 nm) embebidos em matriz de perovskita
Eficiencia: 19.8% (vs 17.2% referencia)
Estabilidade: 75% apos 300h (oxidacao dos QDs)
Custo estimado: Alto (sintese de QDs monodispersos)
Escalabilidade: Baixa (controle de tamanho dos QDs em larga escala)
"""

prompt_comparacao = f"""Analise os tres papers abaixo sobre melhoria de eficiencia
de celulas solares e crie:

1. Uma **tabela comparativa** em Markdown com as colunas:
   Paper | Abordagem | Eficiencia | Ganho Relativo | Estabilidade | Custo | Escalabilidade

2. Uma **analise critica** comparando os trade-offs entre as abordagens.

3. Uma **recomendacao** de qual abordagem e mais promissora para:
   a) Pesquisa academica (proximo paper)
   b) Aplicacao industrial (proximos 5 anos)

Papers:
{papers_comparacao}
"""

comparacao = ask(prompt_comparacao)
print(comparacao)

---

## Identificando Gaps na Literatura

Uma das habilidades mais valiosas na pesquisa e identificar o que
**ainda nao foi feito**. Vamos usar o Claude para analisar criticamente
um campo de pesquisa e sugerir direcoes.

In [ ]:
contexto_pesquisa = """
Campo: Filmes finos de oxidos transparentes condutores (TCOs) para celulas solares

Estado da arte atual:
- ITO (Indium Tin Oxide) e o TCO mais usado, com resistividade ~1e-4 ohm.cm
  e transmitancia >85% no visivel
- O indio e caro e escasso, motivando busca por alternativas
- AZO (Aluminum-doped ZnO) e FTO (Fluorine-doped SnO2) sao alternativas
  mais baratas, mas com desempenho inferior
- Filmes de grafeno e nanotubos de carbono foram testados, mas a resistencia
  de folha ainda e >100 ohm/sq (vs ~10 ohm/sq do ITO)
- Deposicao por sputtering e a tecnica mais comum industrialmente
- Tecnicas de deposicao por solucao (sol-gel, spray pyrolysis) sao mais
  baratas mas produzem filmes de menor qualidade

Papers recentes importantes:
- Kim et al. (2023): AZO com resistividade de 3e-4 ohm.cm por ALD
- Liu et al. (2023): TCO hibrido grafeno/AZO com 20 ohm/sq
- Santos et al. (2022): FTO depositado por spray pyrolysis com 15 ohm/sq
- Nakamura et al. (2023): ITO reciclado de paineis descartados
"""

prompt_gaps = f"""Voce e um pesquisador senior em ciencia de materiais com
20 anos de experiencia em filmes finos e dispositivos fotovoltaicos.

Com base no contexto de pesquisa abaixo, faca uma analise critica e identifique:

1. **Gaps experimentais**: Quais experimentos ou combinacoes de parametros
   ainda nao foram explorados?

2. **Gaps teoricos**: Quais fenomenos fisicos ainda nao sao bem compreendidos?

3. **Gaps metodologicos**: Quais tecnicas de caracterizacao ou fabricacao
   poderiam ser aplicadas mas ainda nao foram?

4. **Direcoes de pesquisa promissoras**: Sugira 3 projetos de pesquisa
   concretos, cada um com:
   - Titulo proposto
   - Hipotese a ser testada
   - Metodologia sugerida
   - Impacto potencial

5. **Perguntas de pesquisa em aberto**: Liste 5 perguntas especificas que
   ainda nao tem resposta na literatura.

Seja critico e especifico. Evite sugestoes genericas.

Contexto:
{contexto_pesquisa}
"""

gaps = ask(prompt_gaps, max_tokens=4096)
print(gaps)

### Refinando a analise de gaps

Apos a primeira analise, podemos pedir ao Claude para aprofundar
em um gap especifico.

In [ ]:
prompt_aprofundar = """Com base na sua analise anterior sobre gaps em TCOs para
celulas solares, aprofunde na direcao mais promissora que voce identificou.

Elabore um plano de pesquisa detalhado incluindo:

1. **Justificativa cientifica** (por que esse gap e importante)
2. **Objetivos especificos** (3-4 objetivos mensuraveis)
3. **Metodologia detalhada**:
   - Materiais e substratos
   - Tecnica de deposicao e parametros
   - Tecnicas de caracterizacao
   - Testes de desempenho
4. **Cronograma estimado** (para um mestrado de 2 anos)
5. **Riscos e planos de contingencia**

Considere que o laboratorio tem acesso a:
- Sputtering RF e DC
- XRD, AFM, SEM
- UV-Vis, Raman
- Efeito Hall
- Simulador solar AM1.5G
"""

plano = ask(prompt_aprofundar)
print(plano)

---

## Criando uma Revisao Bibliografica Estruturada

Organizar multiplos papers em uma narrativa coerente e uma das
partes mais dificeis de escrever uma dissertacao. O Claude pode
ajudar a estruturar o fluxo logico.

In [ ]:
# Resumos de papers para organizar em uma revisao
resumos_papers = """
Paper A - Silva et al. (2021): Revisao historica do desenvolvimento de TCOs
desde os anos 1950. Foco na evolucao das tecnicas de deposicao.

Paper B - Chen et al. (2022): Estudo teorico usando DFT sobre a estrutura
eletronica do ITO e mecanismos de conducao. Mostra que vacancias de oxigenio
sao o principal mecanismo de dopagem.

Paper C - Mueller et al. (2022): Comparacao experimental de ITO, AZO e FTO
depositados por sputtering nas mesmas condicoes. ITO superior em todos os
parametros eletricos, AZO competitivo em transmitancia.

Paper D - Tanaka et al. (2023): Deposicao de AZO por ALD (Atomic Layer
Deposition) com controle de espessura em escala atomica. Resultados
superiores ao sputtering convencional.

Paper E - Rodriguez et al. (2023): Filmes hibridos TCO/grafeno mostrando
sinergia entre conducao do TCO e flexibilidade do grafeno. Aplicacao em
celulas solares flexiveis.

Paper F - Park et al. (2023): Analise de ciclo de vida (LCA) comparando
impacto ambiental de ITO vs alternativas. ITO tem pegada de carbono 5x
maior que AZO.

Paper G - Wang et al. (2023): Machine learning para otimizacao de
parametros de deposicao de TCOs. Reducao de 80% no numero de experimentos
necessarios.
"""

prompt_revisao = f"""Voce e um orientador de mestrado em Engenharia Fisica na UFRGS.
Um aluno te enviou resumos de 7 papers para incluir na revisao bibliografica
da dissertacao sobre "Oxidos Transparentes Condutores para Celulas Solares".

Organize esses papers em uma estrutura logica de revisao bibliografica:

1. Proponha a **estrutura de secoes e subsecoes** da revisao
2. Para cada secao, indique **quais papers** devem ser citados
3. Escreva um **paragrafo de transicao** entre cada secao, mostrando
   como os temas se conectam logicamente
4. Identifique **papers adicionais** que seriam necessarios para
   completar a revisao (descreva o tipo de paper que falta)

Papers disponiveis:
{resumos_papers}
"""

revisao = ask(prompt_revisao, max_tokens=4096)
print(revisao)

---

## System Prompts para Revisor Critico

Diferentes system prompts transformam o Claude em diferentes tipos
de revisores. Isso e util para obter feedback variado sobre seu
trabalho antes de submeter.

In [ ]:
# Texto de exemplo para ser revisado (trecho de um paper ficticio)
trecho_paper = """
The TiO2 thin films were deposited at various temperatures and their properties
were measured. The results show that higher temperature leads to better
crystallinity. The anatase phase was observed at 300 C. We believe this is
because the atoms have more energy to rearrange into the crystalline structure.
The photocatalytic activity was tested using methylene blue degradation and
showed good results. The films deposited at 350 C showed the best performance
with 92% degradation after 3 hours. Statistical analysis was performed using
ANOVA with p < 0.05 considered significant. Three samples were tested for
each condition.
"""

# System prompt 1: Peer Reviewer
system_peer_reviewer = """Voce e um peer reviewer experiente de um journal de
alto impacto (IF > 10) na area de ciencia de materiais. Voce e conhecido
por ser rigoroso mas justo. Analise o texto como se estivesse revisando
um manuscrito submetido. Aponte problemas de:
- Clareza e precisao da linguagem cientifica
- Falta de detalhes experimentais
- Afirmacoes sem suporte adequado
- Estrutura logica do argumento
Forneca sua avaliacao no formato de um parecer de revisao real."""

print("=" * 60)
print("PARECER DO PEER REVIEWER")
print("=" * 60)
review1 = ask(trecho_paper, system=system_peer_reviewer)
print(review1)

In [ ]:
# System prompt 2: Critico de Metodologia
system_metodo_critico = """Voce e um especialista em metodologia experimental
com foco em reproducibilidade e boas praticas de laboratorio. Ao analisar
textos cientificos, voce foca exclusivamente nos aspectos metodologicos:
- Os parametros experimentais estao completos?
- Outro pesquisador conseguiria reproduzir o experimento?
- Ha controles experimentais adequados?
- As condicoes ambientais foram reportadas?
- Os equipamentos e reagentes estao especificados?
Para cada problema encontrado, sugira a informacao especifica que falta."""

print("=" * 60)
print("CRITICA METODOLOGICA")
print("=" * 60)
review2 = ask(trecho_paper, system=system_metodo_critico)
print(review2)

In [ ]:
# System prompt 3: Auditor Estatistico
system_stats_auditor = """Voce e um estatistico consultando para um grupo de
pesquisa em fisica experimental. Sua especialidade e identificar problemas
estatisticos em papers cientificos. Ao analisar textos, foque em:
- Tamanho amostral e poder estatistico
- Escolha adequada dos testes estatisticos
- Reportagem correta de resultados estatisticos
- Barras de erro e incertezas experimentais
- Propagacao de erros
- Vieses de selecao ou confirmacao
- Significancia pratica vs significancia estatistica
Seja especifico sobre o que esta errado e como corrigir."""

print("=" * 60)
print("AUDITORIA ESTATISTICA")
print("=" * 60)
review3 = ask(trecho_paper, system=system_stats_auditor)
print(review3)

### Combinando os tres pareceres

Podemos pedir ao Claude para sintetizar os tres pareceres em um
plano de acao consolidado.

In [ ]:
prompt_consolidar = f"""Voce recebeu tres pareceres diferentes sobre o mesmo
trecho de um paper cientifico. Consolide os feedbacks em um unico plano de
acao priorizado.

Organize em:
1. **Critico (deve corrigir antes de submeter)**: problemas que impediriam publicacao
2. **Importante (deve melhorar)**: problemas que enfraqueceriam o paper
3. **Sugestao (nice to have)**: melhorias que elevariam a qualidade

Para cada item, indique de qual revisor veio e a acao especifica necessaria.

Parecer 1 (Peer Reviewer):
{review1}

Parecer 2 (Critico Metodologico):
{review2}

Parecer 3 (Auditor Estatistico):
{review3}
"""

plano_acao = ask(prompt_consolidar, max_tokens=4096)
print(plano_acao)

---

## Exercicios

### Exercicio 1 - Resumo Estruturado
Encontre um abstract real de um paper na sua area de interesse (pode ser do
Google Scholar ou arXiv). Envie para o Claude e peca um resumo estruturado.
Compare o resumo gerado com sua propria leitura do abstract.

### Exercicio 2 - Extracao de Protocolo
Pegue a secao de Materiais e Metodos de um paper experimental e extraia
um protocolo passo-a-passo. Verifique se alguma informacao critica ficou
faltando no paper original.

### Exercicio 3 - Tabela Comparativa
Selecione 4-5 papers sobre o mesmo tema e crie resumos padronizados.
Use o Claude para gerar uma tabela comparativa e identifique qual
abordagem parece mais promissora.

### Exercicio 4 - Revisao Critica
Escreva um paragrafo descrevendo seus proprios resultados experimentais
(ou inventados). Submeta aos tres revisores criticos (peer reviewer,
critico metodologico, auditor estatistico) e use o feedback para
melhorar o texto.

### Exercicio 5 - Identificacao de Gaps (Desafio)
Escolha um topico de pesquisa do seu interesse. Compile resumos de
10+ papers recentes e use o Claude para identificar gaps e propor
um projeto de pesquisa original. Discuta com seu orientador.

---

## Proximos Passos

Agora que voce sabe usar o Claude para leitura e sintese de papers:

- **Pratique com papers reais** da sua area de pesquisa
- **Crie seus proprios system prompts** especializados para seu campo
- **Combine com ferramentas de busca** (Semantic Scholar API, arXiv API)
  para automatizar a triagem de literatura
- **Mantenha um registro** dos prompts que funcionam melhor para cada tarefa

### Dicas importantes

1. **O Claude nao substitui a leitura critica** - use-o como ferramenta
   de aceleracao, nao como substituto do seu julgamento cientifico
2. **Sempre verifique as informacoes** - o Claude pode inventar detalhes
   ou interpretar incorretamente dados tecnicos
3. **Cuidado com alucinacoes em citacoes** - nunca confie em referencias
   bibliograficas geradas pelo Claude sem verificar
4. **Use iterativamente** - faca perguntas de acompanhamento para
   aprofundar pontos especificos
5. **Documente seus prompts** - bons prompts sao reutilizaveis e
   podem ser compartilhados com colegas de laboratorio

No proximo notebook, vamos explorar como usar o Claude para auxiliar
na **escrita academica** - desde a estruturacao de artigos ate a
revisao de gramatica tecnica em ingles.